In [1]:
import sys
from pathlib import Path
import warnings

PROJECT_ROOT = Path.cwd().parent

warnings.filterwarnings('ignore')
sys.path.append(str(PROJECT_ROOT))

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import pickle

from src.config import (DATA_PATHS, 
                        BIZ,
                        MC_LIGHT,
                        PLT_PARAMS)
from src.data import (load_data, 
                      split_card_data, 
                      split_transactions_by_cards, 
                      fit_transform_scale_features, 
                      transform_scale_features,
                      combine_business_consumer_sets,
                      make_xy)
from src.features import (date_span_days, 
                          build_all_mcc_similarity_features,
                          make_card_features_final,
                          mcc_range_segment)
from src.modeling import (find_threshold_for_min_tpr,
                          collect_model_probabilities,
                          uncertainty_score)

from matplotlib.colors import LinearSegmentedColormap as _LSC
plt.rcParams.update(PLT_PARAMS)
_CM_CMAP = _LSC.from_list('mc_cm', [MC_LIGHT, BIZ])

# Data Loading & Pre-Processing

In [3]:
business, consumer, merchants = load_data(DATA_PATHS)

In [4]:
total_days = date_span_days(pd.concat([business, consumer]))

## Feature Construction

In [5]:
bus_features = make_card_features_final(business, 1, total_days)
consumer_features = make_card_features_final(consumer, 0, total_days)

In [6]:
card_splits = split_card_data(bus_features, consumer_features)

In [7]:
trans_splits = split_transactions_by_cards(business, consumer, card_splits)

(business_mcc_features_train, 
 consumer_mcc_features_train, 
 business_profile, 
 consumer_profile) = build_all_mcc_similarity_features(trans_splits["business"]["train"],
                                                               trans_splits["consumer"]["train"])

(business_mcc_features_val, 
 consumer_mcc_features_val,
 _, _) = build_all_mcc_similarity_features(trans_splits["business"]["val"],
                                                   trans_splits["consumer"]["val"],
                                                   business_profile,
                                                   consumer_profile)

(business_mcc_features_test, 
 consumer_mcc_features_test,
 _, _) = build_all_mcc_similarity_features(trans_splits["business"]["test"],
                                                   trans_splits["consumer"]["test"],
                                                   business_profile,
                                                   consumer_profile)

In [8]:
business_mcc_features = pd.concat([
    business_mcc_features_train, 
    business_mcc_features_val, 
    business_mcc_features_test
])

consumer_mcc_features = pd.concat([
    consumer_mcc_features_train, 
    consumer_mcc_features_val, 
    consumer_mcc_features_test
])

In [9]:
bus_features = bus_features.merge(
    business_mcc_features,
    on = "card_number", 
    how = "left"
)

consumer_features = consumer_features.merge(
    consumer_mcc_features,
    on = "card_number", 
    how = "left"
)

In [10]:
df_splits = split_card_data(bus_features, consumer_features, card_splits = card_splits)

In [11]:
combined_splits = combine_business_consumer_sets(df_splits["business"], df_splits["consumer"])

In [12]:
X, y = make_xy(combined_splits, target_col = "type")

In [13]:
for i in X:
    X[i].drop("card_number", axis = 1, inplace = True)

X["train"]

,online_share,recurring_ratio,repeated_amount_ratio,recurring_it_services_turnover_ratio,recurring_digital_turnover_ratio,share_merchant_country_Kazakhstan,share_merchant_country_US,share_merchant_country_Ireland,active_days_ratio,log_turnover_per_active_day,log_tx_per_active_day,mcc_business_similarity_gap,mcc_avg_distance
0,0.865169,0.067416,0.056180,0.158686,0.269754,0.505091,0.266643,0.000000,0.576923,11.428673,0.991487,0.021479,0.572035
1,0.891892,0.040541,0.033784,0.160647,0.160647,0.715180,0.275281,0.000000,0.494505,11.888500,0.972461,0.096675,0.654736
2,0.925000,0.150000,0.125000,0.290570,0.426691,0.512817,0.316392,0.144744,0.439560,12.233194,0.916291,0.172482,0.438594
3,0.870130,0.038961,0.032468,0.118540,0.118540,0.768027,0.137362,0.000000,0.510989,12.227733,0.976789,0.029696,0.383201
4,0.781818,0.163636,0.136364,0.169093,0.227021,0.610902,0.180040,0.000000,0.417582,12.144697,0.895013,0.099216,0.349801
...,...,...,...,...,...,...,...,...,...,...,...,...,...
62995,0.542986,0.054299,0.045249,0.003050,0.025715,0.401011,0.059943,0.539046,0.681319,12.131302,1.023263,0.231059,0.493682
62996,0.607477,0.000000,0.000000,0.000000,0.000000,0.516619,0.041039,0.000000,0.412088,11.675676,0.886519,-0.174019,0.459430
62997,0.525424,0.101695,0.084746,0.000000,0.009808,0.743788,0.021895,0.000000,0.269231,11.138515,0.790311,-0.056356,0.342970
62998,0.455446,0.059406,0.049505,0.000000,0.099268,0.163838,0.002235,0.150890,0.357143,11.030232,0.937601,-0.201376,0.313016


In [14]:
X_train = X["train"]
X_val = X["val"]
X_test = X["test"]

y_train = y["train"]
y_val = y["val"]
y_test = y["test"]

## Scaling

In [15]:
features_to_scale = [
    "log_tx_per_active_day",
    "log_turnover_per_active_day"
]

In [16]:
X_train, scaler = fit_transform_scale_features(X_train, features_to_scale)
X_val = transform_scale_features(X_val, features_to_scale, scaler)
X_test = transform_scale_features(X_test, features_to_scale, scaler)

# Model Import

In [17]:
logreg = pickle.load(open("../models/logistic_regression.pkl", "rb"))
knn = pickle.load(open("../models/knn.pkl", "rb"))
svm = pickle.load(open("../models/svm.pkl", "rb"))
rf = pickle.load(open("../models/random_forest.pkl", "rb"))
xgboost = pickle.load(open("../models/xgboost.pkl", "rb"))

models = [logreg, knn, svm, rf, xgboost]

# Combining the Models

First of all let us find a threshold for minimal TPR that will be used to calculate the model uncertainty.

In [18]:
model_probs = collect_model_probabilities(models, X_val).mean(axis = 1)
threshold = find_threshold_for_min_tpr(y_val, model_probs)

print(f"obtained threshold: {threshold}")

obtained threshold: 0.19


In [19]:
total_df = pd.concat([combined_splits["train"], combined_splits["val"], combined_splits["test"]])
models_df = transform_scale_features(total_df.drop(["card_number", "type"], axis = 1), features_to_scale, scaler)

In [20]:
probs = collect_model_probabilities(models, models_df)
mean_prob, uncertainty = uncertainty_score(probs, threshold)

result_df = pd.DataFrame({
    "card_number": total_df["card_number"],
    "probability": mean_prob,
    "uncertainty": uncertainty,
    "prediction": (mean_prob >= threshold).astype(int)
})

In [21]:
result_df

,card_number,probability,uncertainty,prediction
0,5100610003025081,0.917084,0.029568,1
1,5100610003860784,0.935472,0.014998,1
2,5100610013466473,0.977581,0.002685,1
3,5100610016135521,0.964485,0.005753,1
4,5100610030226389,0.967885,0.001851,1
...,...,...,...,...
20995,5531519987359573,0.004811,0.000214,0
20996,5531519991198009,0.004813,0.000214,0
20997,5531519991627445,0.004800,0.000214,0
20998,5531519993305461,0.005171,0.000248,0


# Card Features for Dashboard

In [22]:
dashboard_df = pd.concat([business, consumer])

dashboard_df["transaction_date"] = pd.to_datetime(dashboard_df["transaction_date"], errors="coerce")
dashboard_df["mcc_range_segment"] = dashboard_df["mcc"].apply(mcc_range_segment)
dashboard_df["is_online"] = (dashboard_df["channel"] == "online").astype(int)

card_features = (
    dashboard_df.groupby("card_number")
    .agg(
        transaction_count=("transaction_amount_kzt", "size"),
        total_turnover=("transaction_amount_kzt", "sum"),
        avg_transaction_amount=("transaction_amount_kzt", "mean"),
        online_share=("is_online", "mean"),
        recurring_ratio=("is_recurring", "mean"),
        merchant_diversity=("merchant_id", "nunique"),
        mcc_diversity=("mcc", "nunique"),
        last_transaction_date=("transaction_date", "max"),
    )
    .reset_index()
)

top_mcc_group = (
    dashboard_df.groupby(["card_number", "mcc_range_segment"])
    .size()
    .reset_index(name="count")
    .sort_values(["card_number", "count"], ascending=[True, False])
    .drop_duplicates("card_number")
    [["card_number", "mcc_range_segment"]]
    .rename(columns={"mcc_range_segment": "top_mcc_group"})
)

top_country = (
    dashboard_df.groupby(["card_number", "country"])
    .size()
    .reset_index(name="count")
    .sort_values(["card_number", "count"], ascending=[True, False])
    .drop_duplicates("card_number")
    [["card_number", "country"]]
    .rename(columns={"country": "top_country"})
)

card_features = (
    card_features
    .merge(top_mcc_group, on="card_number", how="left")
    .merge(top_country, on="card_number", how="left")
)

card_features = card_features[
    [
        "card_number",
        "transaction_count",
        "total_turnover",
        "avg_transaction_amount",
        "online_share",
        "recurring_ratio",
        "merchant_diversity",
        "mcc_diversity",
        "top_mcc_group",
        "top_country",
        "last_transaction_date",
    ]
]

card_features

,card_number,transaction_count,total_turnover,avg_transaction_amount,online_share,recurring_ratio,merchant_diversity,mcc_diversity,top_mcc_group,top_country,last_transaction_date
0,5100610003025081,178,9651486,54221.831461,0.865169,0.067416,9,9,misc_stores,Kazakhstan,2026-03-31
1,5100610003044611,106,14353608,135411.396226,0.905660,0.169811,18,18,transportation,Kazakhstan,2026-03-31
2,5100610003860784,148,13102356,88529.432432,0.891892,0.040541,8,8,transportation,Kazakhstan,2026-03-31
3,5100610005930965,43,14215001,330581.418605,0.883721,0.418605,7,7,business_services,Kazakhstan,2026-03-31
4,5100610005962109,119,1303168,10950.991597,0.512605,0.050420,46,43,travel_private,Kazakhstan,2026-03-31
...,...,...,...,...,...,...,...,...,...,...,...
104995,5531519993295407,88,1955744,22224.363636,0.443182,0.068182,32,28,misc_stores,Kazakhstan,2026-03-30
104996,5531519993305461,79,8578216,108585.012658,0.569620,0.000000,10,10,retail_outlets,Kazakhstan,2026-03-30
104997,5531519994366645,74,2412818,32605.648649,0.378378,0.000000,27,23,clothing,Kazakhstan,2026-03-30
104998,5531519994943930,111,2597711,23402.801802,0.522523,0.054054,35,31,travel_private,Kazakhstan,2026-03-31


In [23]:
df_to_export = card_features.merge(
    result_df,
    how = "inner",
    on = "card_number"
)

df_to_export.to_csv("../dashboard/dashboard_data.csv", index = False)